# Прогноз зарплаты: явные признаки, честная валидация и интеграция модели

Ноутбук собирает модель прогноза зарплаты по вакансии. В отличие от прежней версии, X **не** получается удалением таргета из всех колонок: список признаков зафиксирован в `src/features.py` по результатам EDA и `hypothesis_testing.ipynb`.

**Таргет:** `log_salary = log(salary_mid)`, где `salary_mid` — середина вилки в USD за год.

**Что важно помнить про логарифм:** модель предсказывает логарифм, а `np.exp(prediction)` возвращает оценку, близкую к условной медиане, а не гарантированное условное среднее. Качество поэтому оценивается и в лог-шкале, и в USD.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, cross_val_predict, train_test_split

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import load_data
from src.preprocessing import prepare_salary_data
from src.features import (
    BASE_FEATURES,
    CATEGORICAL_FEATURES,
    EXCLUDED_COLUMNS,
    ID_COLUMNS,
    LEAKAGE_COLUMNS,
    NUMERIC_FEATURES,
    RANDOM_STATE,
    TARGET_ENCODING_FEATURES,
    TARGET_LOG,
    TARGET_USD,
    TEXT_COLUMN,
    TEXT_MASKED_COLUMN,
    add_engineered_features,
    assert_no_leakage,
    describe_feature_policy,
    feature_frame,
)
from src.modeling import (
    OOFTargetEncoder,
    SalaryModel,
    compute_smearing_factor,
    evaluate,
    make_model,
    make_preprocessor,
    make_text_model,
    predictions_to_usd,
    save_model,
)

## 1. Подготовка данных

Используем общий модуль `src/preprocessing.py`: пересчёт в USD/год, таргет, флаги некорректных значений. Затем `src/features.py` добавляет только инженерные признаки, разрешённые для модели.

In [2]:
raw = load_data()
df = prepare_salary_data(raw)
df = add_engineered_features(df)

df_model = df.loc[df[TARGET_LOG].notna()].copy()

print(f"Исходных строк: {len(raw):,}")
print(f"Строк с валидным таргетом: {len(df_model):,}")
print(f"Компаний: {df_model['company_id'].nunique():,}")
print(f"Пропусков в таргете: {df_model[TARGET_LOG].isna().sum()}")
display(df_model[[TARGET_USD, TARGET_LOG, "salary_min", "salary_max", "raw_period"]].describe().round(3))

Исходных строк: 9,173
Строк с валидным таргетом: 9,155
Компаний: 4,166
Пропусков в таргете: 0


,salary_mid,log_salary,salary_min,salary_max
count,9155.000,9155.000,9155.000,9155.000
mean,116837.223,11.479,100619.739,133054.707
std,80385.401,0.621,69480.956,93522.262
min,2307.692,7.744,2051.282,2564.103
25%,60000.000,11.002,52000.000,66901.408
50%,99500.000,11.508,85000.000,112406.000
75%,155000.000,11.951,131000.000,178940.000
max,1226966.292,14.020,1168539.326,1380000.000


## 2. Явный список признаков

В модель попадают только колонки из `BASE_FEATURES`; `company_id` — только через target encoding. Проверяем, что `job_id`, `description` и все производные таргета отсутствуют.

In [3]:
print("Категориальные признаки:")
print(CATEGORICAL_FEATURES)
print("\nЧисловые признаки:")
print(NUMERIC_FEATURES)
print("\nTarget encoding:")
print(TARGET_ENCODING_FEATURES)

display(describe_feature_policy())

X_all = feature_frame(df_model, include_te=True)
assert_no_leakage(X_all.columns)

forbidden_present = sorted(set(X_all.columns) & set(ID_COLUMNS + LEAKAGE_COLUMNS + [TEXT_COLUMN]))
assert not forbidden_present, forbidden_present
print(f"\nФорма X: {X_all.shape}")
print("Запрещённых колонок в X нет: job_id, description, salary_* и производные таргета исключены.")

Категориальные признаки:
['raw_period', 'ats', 'country_code', 'employment_type', 'workplace_type', 'seniority', 'seniority_title', 'family', 'department', 'us_state']

Числовые признаки:
['description_len', 'is_missing_workplace_type', 'is_missing_seniority', 'is_missing_language', 'is_missing_country_code', 'is_missing_employment_type', 'is_missing_department', 'is_missing_region', 'is_missing_city']

Target encoding:
['company_id']


,признак,роль,источник
0,raw_period,категориальный,EDA / hypothesis_testing
1,ats,категориальный,EDA / hypothesis_testing
2,country_code,категориальный,EDA / hypothesis_testing
3,employment_type,категориальный,EDA / hypothesis_testing
4,workplace_type,категориальный,EDA / hypothesis_testing
5,seniority,категориальный,EDA / hypothesis_testing
6,seniority_title,категориальный,EDA / hypothesis_testing
7,family,категориальный,EDA / hypothesis_testing
8,department,категориальный,EDA / hypothesis_testing
9,us_state,категориальный,EDA / hypothesis_testing



Форма X: (9155, 20)
Запрещённых колонок в X нет: job_id, description, salary_* и производные таргета исключены.


## 3. Две схемы валидации

1. Обычный `train_test_split` — оценка «в знакомых компаниях»: компания может встретиться и в train, и в test.
2. `GroupShuffleSplit` по `company_id` — оценка обобщения на **новые** компании: группы не пересекаются.

Обе схемы используют `random_state=42`, поэтому сравнение воспроизводимо.

In [4]:
X = X_all
y = df_model[TARGET_LOG]
groups = df_model["company_id"]
y_usd = df_model[TARGET_USD]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
group_train_idx, group_test_idx = next(splitter.split(X, y, groups=groups))
Xg_train, Xg_test = X.iloc[group_train_idx], X.iloc[group_test_idx]
yg_train, yg_test = y.iloc[group_train_idx], y.iloc[group_test_idx]

company_overlap = set(groups.iloc[group_train_idx]) & set(groups.iloc[group_test_idx])
print(f"Случайный split: train={len(X_train):,}, test={len(X_test):,}")
print(f"Group split: train={len(Xg_train):,}, test={len(Xg_test):,}")
print(f"Пересечение компаний в group split: {len(company_overlap)}")

Случайный split: train=7,324, test=1,831
Group split: train=7,367, test=1,788
Пересечение компаний в group split: 0


## 4. Метрики

- **MAE (log)** — основная метрика сравнения: ошибка в лог-шкале, примерно средняя относительная ошибка.
- **MAE (USD)** — ошибка после `exp`, в долларах; чувствительна к дорогим хвостам и смещению обратного преобразования.
- **MAPE** и **RMSLE** — дополнительные метрики исходной шкалы.

`exp(MAE_log) - 1` — удобное приближение относительной ошибки.

In [5]:
def evaluate_model(model, X_eval, y_eval_log, label, *, smearing_factor=1.0):
    pred_log = pd.Series(model.predict(X_eval), index=X_eval.index, name=label)
    actual_usd = df_model.loc[X_eval.index, TARGET_USD]
    metrics = evaluate(y_eval_log, pred_log, actual_usd, smearing_factor=smearing_factor)
    metrics["MAE (log), %"] = (np.exp(metrics["MAE (log)"]) - 1) * 100
    return pd.Series(metrics), pred_log

## 5. Базовая модель и три регрессора

Все модели обучаются на одном X и одном split. `DummyRegressor` предсказывает медиану train — это нижняя граница качества.

In [6]:
standard_predictions = {}
standard_rows = {}
standard_models = {}

for key, label in [
    ("dummy", "DummyRegressor (median)"),
    ("linear", "LinearRegression"),
    ("random_forest", "RandomForestRegressor"),
    ("hist_gradient_boosting", "HistGradientBoostingRegressor"),
]:
    model = make_model(key).fit(X_train, y_train)
    smearing = compute_smearing_factor(model, X_train, y_train) if key != "dummy" else 1.0
    row, pred = evaluate_model(model, X_test, y_test, label, smearing_factor=smearing)
    standard_models[key] = model
    standard_predictions[key] = pred
    standard_rows[label] = row
    print(f"{label}: MAE(log)={row['MAE (log)']:.4f}, MAE(USD)={row['MAE (USD)']:,.0f}")

standard_table = pd.DataFrame(standard_rows).T
display(standard_table.round(3))

DummyRegressor (median): MAE(log)=0.5023, MAE(USD)=54,424


LinearRegression: MAE(log)=0.2788, MAE(USD)=33,690


RandomForestRegressor: MAE(log)=0.2826, MAE(USD)=33,305


HistGradientBoostingRegressor: MAE(log)=0.2737, MAE(USD)=32,279


,MAE (log),RMSE (log),R2 (log),MAE (USD),RMSE (USD),MAPE,RMSLE,"MAE (log), %"
DummyRegressor (median),0.502,0.607,-0.001,54423.566,77968.257,0.579,0.607,65.244
LinearRegression,0.279,0.372,0.624,33689.969,57755.039,0.328,0.380,32.160
RandomForestRegressor,0.283,0.380,0.608,33305.230,56801.011,0.323,0.385,32.652
HistGradientBoostingRegressor,0.274,0.368,0.632,32279.195,55556.947,0.310,0.372,31.482


## 6. Устойчивость к новым компаниям

Повторяем обучение на `GroupShuffleSplit` по `company_id`. Если MAE заметно растёт, модель частично запоминает работодателей, а не общие закономерности рынка. Это ожидаемо для target encoding компании, но масштаб разрыва нужно измерять.

In [7]:
group_predictions = {}
group_rows = {}
group_models = {}

for key, label in [
    ("dummy", "DummyRegressor (median)"),
    ("linear", "LinearRegression"),
    ("random_forest", "RandomForestRegressor"),
    ("hist_gradient_boosting", "HistGradientBoostingRegressor"),
]:
    model = make_model(key).fit(Xg_train, yg_train)
    smearing = compute_smearing_factor(model, Xg_train, yg_train) if key != "dummy" else 1.0
    row, pred = evaluate_model(model, Xg_test, yg_test, label, smearing_factor=smearing)
    group_models[key] = model
    group_predictions[key] = pred
    group_rows[label] = row

group_table = pd.DataFrame(group_rows).T

comparison = pd.DataFrame({
    "MAE (log), random split": standard_table["MAE (log)"],
    "MAE (log), group split": group_table["MAE (log)"],
    "Δ MAE (log)": group_table["MAE (log)"] - standard_table["MAE (log)"],
    "MAE (USD), random split": standard_table["MAE (USD)"],
    "MAE (USD), group split": group_table["MAE (USD)"],
    "Δ MAE (USD)": group_table["MAE (USD)"] - standard_table["MAE (USD)"],
})
display(comparison.round(3))

,"MAE (log), random split","MAE (log), group split",Δ MAE (log),"MAE (USD), random split","MAE (USD), group split",Δ MAE (USD)
DummyRegressor (median),0.502,0.523,0.021,54423.566,55407.846,984.279
LinearRegression,0.279,0.318,0.040,33689.969,35299.228,1609.259
RandomForestRegressor,0.283,0.309,0.027,33305.230,34483.261,1178.032
HistGradientBoostingRegressor,0.274,0.306,0.032,32279.195,34143.006,1863.811


`GroupShuffleSplit` даёт одну оценку и может «повезти» с составом компаний. Для проверки стабильности дополнительно считаем 5-fold `GroupKFold` для линейной модели: каждая компания целиком попадает в один фолд.

In [8]:
group_kfold = GroupKFold(n_splits=5)
oof_group = cross_val_predict(
    make_model("linear"), X, y, groups=groups, cv=group_kfold, n_jobs=-1
)
group_cv_metrics = evaluate(y, oof_group, y_usd)
group_cv_metrics["MAE (log), %"] = (np.exp(group_cv_metrics["MAE (log)"]) - 1) * 100
display(pd.Series(group_cv_metrics).to_frame("GroupKFold, LinearRegression").round(3))

,"GroupKFold, LinearRegression"
MAE (log),0.312
RMSE (log),0.416
R2 (log),0.552
MAE (USD),35624.332
RMSE (USD),65998.542
MAPE,0.336
RMSLE,0.416
"MAE (log), %",36.623


## 7. Корректность Target Encoding

`OOFTargetEncoder` делает cross-fitting:

- для каждой обучающей строки статистика компании считается по **другим** фолдам, поэтому таргет самой строки не участвует в её признаке;
- для validation/test используется статистика, рассчитанная **только на полном train**;
- неизвестная компания получает prior train.

Проверим свойство напрямую: у компаний по несколько строк, поэтому OOF-признак должен строиться по другим строкам компании. Если изменить таргет одной строки, её собственный OOF-признак меняться не должен, а `transform` на полном train — должен.


In [9]:
# У каждой компании несколько строк: OOF-признак строки должен считаться
# по другим фолдам, а не по её собственному таргету.
probe = pd.DataFrame({
    "company_id": ["alpha"] * 4 + ["beta"] * 4 + ["gamma"] * 4,
    "row": np.arange(12),
})
probe_target = pd.Series(np.linspace(10.0, 12.0, len(probe)))

probe_encoder = OOFTargetEncoder(columns=["company_id"], smoothing=0.0,
                                 n_splits=4, random_state=RANDOM_STATE)
probe_oof = probe_encoder.fit_transform(probe[["company_id"]], probe_target)
probe_full = probe_encoder.transform(probe[["company_id"]])

changed_target = probe_target.copy()
changed_target.iloc[0] += 100
changed_encoder = OOFTargetEncoder(columns=["company_id"], smoothing=0.0,
                                   n_splits=4, random_state=RANDOM_STATE)
changed_oof = changed_encoder.fit_transform(probe[["company_id"]], changed_target)
changed_full = changed_encoder.transform(probe[["company_id"]])

display(pd.DataFrame({
    "company_id": probe["company_id"],
    "таргет строки": probe_target,
    "OOF-признак": probe_oof["company_id_te"],
    "transform на train": probe_full["company_id_te"],
}).head(8))

print("OOF-признак 1-й строки не изменился после изменения её таргета:",
      bool(np.isclose(probe_oof["company_id_te"].iloc[0],
                      changed_oof["company_id_te"].iloc[0])))
print("Полная статистика train при этом изменилась:",
      bool(not np.isclose(probe_full["company_id_te"].iloc[0],
                          changed_full["company_id_te"].iloc[0])))


,company_id,таргет строки,OOF-признак,transform на train
0,alpha,10.000000,10.363636,10.272727
1,alpha,10.181818,10.303030,10.272727
2,alpha,10.363636,10.242424,10.272727
3,alpha,10.545455,10.181818,10.272727
4,beta,10.727273,11.090909,11.000000
5,beta,10.909091,11.030303,11.000000
6,beta,11.090909,10.818182,11.000000
7,beta,11.272727,10.818182,11.000000


OOF-признак 1-й строки не изменился после изменения её таргета: True
Полная статистика train при этом изменилась: True


## 8. Отдельный текстовый пайплайн

`description` не входит в базовые модели: OneHotEncoder по сырому тексту создал бы тысячи категорий. Вместо этого — отдельный `TfidfVectorizer` + Ridge. Суммы в описании маскируются, чтобы модель не читала готовую зарплату.

Сравниваем три варианта на одном split: метаданные, только текст, метаданные + текст.

In [10]:
X_text = X.copy()
X_text[TEXT_MASKED_COLUMN] = df_model.loc[X_text.index, TEXT_MASKED_COLUMN]
text_train = X_text.loc[X_train.index]
text_test = X_text.loc[X_test.index]

text_rows = {}
text_predictions = {}

# 1. Только текст
text_model = make_text_model().fit(text_train[TEXT_MASKED_COLUMN], y_train)
row, pred = evaluate_model(
    text_model, text_test[TEXT_MASKED_COLUMN].fillna(""), y_test, "TF-IDF only"
)
text_rows["TF-IDF only"] = row
text_predictions["text_only"] = pred

# 2. Метаданные + текст
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

combined_model = Pipeline([
    ("features", ColumnTransformer([
        ("meta", make_preprocessor(dense=False), BASE_FEATURES + TARGET_ENCODING_FEATURES),
        ("text", TfidfVectorizer(max_features=8000, min_df=5, stop_words="english"), TEXT_MASKED_COLUMN),
    ])),
    ("regressor", Ridge(alpha=1.0)),
])
combined_model.fit(text_train, y_train)
row, pred = evaluate_model(combined_model, text_test, y_test, "metadata + TF-IDF")
text_rows["metadata + TF-IDF"] = row
text_predictions["combined"] = pred

text_table = pd.DataFrame(text_rows).T
base_linear = standard_table.loc["LinearRegression", ["MAE (log)", "MAE (USD)", "MAPE"]]
text_table.loc["metadata only (LinearRegression)", ["MAE (log)", "MAE (USD)", "MAPE"]] = base_linear
text_table["Δ MAE (log) к метаданным, %"] = (
    text_table["MAE (log)"] / base_linear["MAE (log)"] - 1
) * 100
display(text_table.round(3))

,MAE (log),RMSE (log),R2 (log),MAE (USD),RMSE (USD),MAPE,RMSLE,"MAE (log), %","Δ MAE (log) к метаданным, %"
TF-IDF only,0.252,0.333,0.698,29407.009,51883.647,0.269,0.333,28.711,-9.483
metadata + TF-IDF,0.222,0.302,0.752,26478.276,49204.726,0.236,0.302,24.914,-20.222
metadata only (LinearRegression),0.279,NaN,NaN,33689.969,NaN,0.328,NaN,NaN,0.000


## 9. Что именно означает `exp(prediction)`

Модель минимизирует MAE в лог-шкале. При примерно нормальных остатках логарифма `exp(pred_log)` — оценка условной **медианы** зарплаты. Условное среднее в исходной шкале равно `exp(pred_log) * E[exp(остаток)]`; множитель Duan's smearing оценивается на train как `mean(exp(residual))`.

Поэтому:

- `MAE (log)` — честная основная метрика;
- `MAE (USD)` считается после `exp` и показывает практическую цену ошибки;
- `exp(pred)` нельзя называть гарантированным средним: это медиана/типичное значение, а smearing — лишь поправка при дополнительных допущениях.

In [11]:
best_label = standard_table["MAE (log)"].idxmin()
label_to_key = {
    "DummyRegressor (median)": "dummy",
    "LinearRegression": "linear",
    "RandomForestRegressor": "random_forest",
    "HistGradientBoostingRegressor": "hist_gradient_boosting",
}
best_key = label_to_key[best_label]
best_model = standard_models[best_key]
best_smearing = compute_smearing_factor(best_model, X_train, y_train)

pred_log = standard_predictions[best_key]
pred_exp = predictions_to_usd(pred_log)
pred_smearing = predictions_to_usd(pred_log, smearing_factor=best_smearing)
actual_usd = df_model.loc[X_test.index, TARGET_USD]

back_transform = pd.DataFrame({
    "exp(pred)": {
        "среднее, USD": pred_exp.mean(),
        "медиана, USD": np.median(pred_exp),
        "MAE, USD": np.mean(np.abs(actual_usd - pred_exp)),
    },
    "exp(pred) * smearing": {
        "среднее, USD": pred_smearing.mean(),
        "медиана, USD": np.median(pred_smearing),
        "MAE, USD": np.mean(np.abs(actual_usd - pred_smearing)),
    },
    "факт": {
        "среднее, USD": actual_usd.mean(),
        "медиана, USD": actual_usd.median(),
        "MAE, USD": 0.0,
    },
})
print(f"Лучшая базовая модель по MAE(log): {best_label}")
print(f"Duan smearing factor: {best_smearing:.4f}")
display(back_transform.round(0))

Лучшая базовая модель по MAE(log): HistGradientBoostingRegressor
Duan smearing factor: 1.0419


,exp(pred),exp(pred) * smearing,факт
"среднее, USD",111577.0,116253.0,116743.0
"медиана, USD",105119.0,109525.0,99665.0
"MAE, USD",31813.0,32279.0,0.0


## 10. Итоговая модель и интеграция

Сохраняем лучшую **табличную** модель (без `description`) в `models/salary_model.joblib`. Она принимает «сырую» запись о вакансии через `src.predict.predict_salary`; текстовый вариант остаётся отдельным экспериментом, потому что для него обязателен `description`.

Артефакт содержит пайплайн, список признаков, smearing-коэффициент и метаданные эксперимента.

In [12]:
model_path = PROJECT_ROOT / "models" / "salary_model.joblib"
model_path.parent.mkdir(parents=True, exist_ok=True)

final_pipeline = make_model(best_key).fit(X, y)
final_smearing = compute_smearing_factor(final_pipeline, X, y)

bundle = SalaryModel(
    pipeline=final_pipeline,
    feature_columns=list(X.columns),
    smearing_factor=final_smearing,
    metadata={
        "best_model": best_label,
        "target": "log(salary_mid), salary_mid in USD/year",
        "random_state": RANDOM_STATE,
        "features": list(X.columns),
        "excluded": EXCLUDED_COLUMNS,
        "test_metrics_random_split": standard_table.loc[best_label].to_dict(),
        "test_metrics_group_split": group_table.loc[best_label].to_dict(),
        "group_kfold_linear": group_cv_metrics,
        "text_experiment": text_table.to_dict(),
        "fx_rates_as_of": "2026-10-15",
    },
)
save_model(bundle, model_path)
print(f"Модель сохранена: {model_path}")
print("Признаки:", list(X.columns))

Модель сохранена: D:\Job Postings with Salaries\models\salary_model.joblib
Признаки: ['raw_period', 'ats', 'country_code', 'employment_type', 'workplace_type', 'seniority', 'seniority_title', 'family', 'department', 'us_state', 'description_len', 'is_missing_workplace_type', 'is_missing_seniority', 'is_missing_language', 'is_missing_country_code', 'is_missing_employment_type', 'is_missing_department', 'is_missing_region', 'is_missing_city', 'company_id']


In [13]:
from src.predict import predict_salary

example = {
    "title": "Senior Data Scientist",
    "company_id": "new-company",
    "ats": "greenhouse",
    "country_code": "US",
    "employment_type": "full-time",
    "workplace_type": "remote",
    "seniority": None,
    "department": "Data",
    "location": "New York, NY",
    "raw_period": "year",
    "description": "Build ML models and dashboards.",
}
prediction = predict_salary(example, model_path)
display(pd.DataFrame([prediction]))

,salary_pred_log,salary_pred_usd,salary_pred_usd_conditional_mean,note
0,11.807793,134294.784573,134294.784573,salary_pred_usd = exp(log_prediction) — оценка...


## 11. Выводы и ограничения

- Признаки заданы явно; `job_id`, `description`, `salary_min`, `salary_max`, `salary_mid`, `log_salary`, annual-колонки, `salary_text`, `spread` и флаги выбросов в X не попадают.
- `company_id` используется только через out-of-fold target encoding; при разделении по компаниям качество ниже, чем на случайном split.
- `description` вынесен в отдельный TF-IDF-пайплайн; он даёт сильный прирост, но требует текста и маскирования сумм.
- Основная метрика — MAE в лог-шкале; MAE в USD и MAPE зависят от обратного преобразования и дорогих хвостов.
- Данные — вакансии с опубликованной зарплатой за короткий период; возможны повторные публикации, а зарплаты приведены к USD без поправки на налоги и стоимость жизни.